# 9.8 · 学习率调度 / Learning Rate Schedulers

> **课程定位 / Where this fits**
> 第 8 课，**Part 9 · 深度学习基础**。
> Lesson 8, **Part 9 · Deep Learning Foundations**.
>
> 9.7 的优化器都有一个固定学习率，但**最好的学习率不是一成不变的**：训练初期想用大学习率快速下降，后期想用小学习率精细收敛。**学习率调度(LR scheduler)** 就是按训练进度动态调整学习率。它常常是"白训"和"训出好模型"的分界线——**warmup、cosine、one-cycle** 是现代训练的标配。
> The optimizers in 9.7 use a fixed LR, but **the best LR isn't constant**: early on you want a large LR to descend fast, later a small LR for fine convergence. **LR schedulers** adjust the LR over training. They're often the line between "wasted training" and "a good model" — **warmup, cosine, one-cycle** are standard in modern training.
>
> 💼 **实战/面试视角**："为什么要学习率衰减 / warmup 干什么 / cosine/one-cycle" 是训练调优常考。
> 💼 **Practical/interview angle:** "why LR decay / what warmup does / cosine/one-cycle" — training-tuning questions.

> 📐 **符号约定 / Notation**
> - $\eta_t$ —— 第 $t$ 步的学习率 / LR at step $t$
> - warmup —— 训练初期从小到大逐步升 LR / gradual LR ramp-up at the start

> 💡 **面试相关 / Interview-relevant**
> - "为什么需要学习率衰减"（出镜率 ★★★★★）
> - "warmup 解决什么问题"（出镜率 ★★★★，初期不稳）
> - "cosine annealing 是什么"（★★★★）
> - "ReduceLROnPlateau 怎么用"（★★★）
> - "one-cycle policy"（★★★）

---

## 学习目标 / Learning Objectives

1. 理解为什么固定学习率不够、需要衰减。
   Understand why a fixed LR isn't enough and decay is needed.
2. 掌握 **step / 指数 / cosine** 衰减。
   Master step / exponential / cosine decay.
3. 理解 **warmup** 为何能稳住训练初期。
   Understand why warmup stabilizes early training.
4. 用 **ReduceLROnPlateau** 自适应降 LR。
   Use ReduceLROnPlateau for adaptive LR drops.
5. 了解 **one-cycle** 策略并对比效果。
   Know the one-cycle policy and compare its effect.

## 目录 / TOC
1. [先建直觉：为什么要调度 ⭐](#1)
2. [常见调度曲线（可视化）⭐](#2)
3. [Warmup + Cosine ⭐](#3)
4. [ReduceLROnPlateau ⭐](#4)
5. [实测：调度 vs 固定 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉：为什么要调度 ⭐ / Intuition: Why Schedule

想象下山找最低点：**刚开始离谷底远，应该大步走（大学习率）快速接近**；**快到谷底时，应该小步走（小学习率），否则会在最低点附近反复横跳、停不下来**。固定学习率没法同时满足这两个需求——要么初期太慢，要么后期不收敛。
Picture descending to the lowest point: **far from the bottom, take big steps (large LR) to approach fast**; **near the bottom, take small steps (small LR), or you'll bounce around the minimum forever**. A fixed LR can't do both — either too slow early or never converging late.

**学习率衰减(LR decay)** 解决它：**初期大、后期小**。这是几乎所有深度学习训练的标准做法。再加两个现代技巧：**warmup**（一开始反而先慢慢升，稳住初期）和 **cosine/one-cycle**（更平滑的形状）。
**LR decay** fixes this: **large early, small late**. Standard in almost all DL training. Plus two modern tricks: **warmup** (ramp up slowly at first to stabilize the start) and **cosine/one-cycle** (smoother shapes).


<a id="2"></a>
## 2. 常见调度曲线（可视化）⭐ / Common Schedule Curves

先把几种最常用的调度的**学习率曲线**画出来，直观感受形状（横轴是训练步/epoch，纵轴是学习率）：
Let's plot the **LR curves** of common schedules to feel their shapes (x = training step/epoch, y = LR):
- **Step decay（阶梯）**：每隔若干 epoch 把 LR 乘以一个因子（如每 30 轮 ×0.1）。简单粗暴。
  **Step decay:** multiply LR by a factor every few epochs (e.g. ×0.1 every 30). Simple and blunt.
- **Exponential（指数）**：每步 ×一个常数，平滑指数下降。
  **Exponential:** multiply by a constant each step, smooth exponential decay.
- **Cosine annealing（余弦退火）**：按半个余弦曲线从大平滑降到小。现代训练最常用之一。
  **Cosine annealing:** follow a half-cosine from large to small. One of the most popular today.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn
sns.set_theme(style="whitegrid")

epochs = 100; base_lr = 0.1
# 用 PyTorch 的调度器生成各曲线 / generate curves with torch schedulers
def get_lrs(scheduler_fn):
    net = nn.Linear(1,1); opt = torch.optim.SGD(net.parameters(), lr=base_lr)
    sched = scheduler_fn(opt); lrs = []
    for _ in range(epochs):
        lrs.append(opt.param_groups[0]["lr"]); opt.step(); sched.step()
    return lrs

curves = {
    "Step (每30轮×0.1)": lambda o: torch.optim.lr_scheduler.StepLR(o, step_size=30, gamma=0.1),
    "Exponential (×0.97)": lambda o: torch.optim.lr_scheduler.ExponentialLR(o, gamma=0.97),
    "Cosine annealing": lambda o: torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=epochs),
}
fig, ax = plt.subplots(figsize=(8, 4))
for name, fn in curves.items():
    ax.plot(get_lrs(fn), label=name, lw=2)
ax.axhline(base_lr, color="gray", ls="--", label="固定 LR (无调度)")
ax.set_xlabel("epoch"); ax.set_ylabel("学习率 LR"); ax.legend(); ax.set_title("常见学习率调度曲线: 初期大, 后期小")
plt.tight_layout(); plt.show()
print("Step: 阶梯式骤降; Exponential: 平滑指数降; Cosine: 半余弦平滑降(现代最常用之一)")
print("共同点: 初期大学习率快速下降, 后期小学习率精细收敛")


<a id="3"></a>
## 3. Warmup + Cosine ⭐ / Warmup + Cosine

**Warmup（预热）** 是个反直觉但极有用的技巧：训练**最开始的几百步，先把学习率从很小逐渐升到目标值**，而不是一上来就用大 LR。
**Warmup** is a counterintuitive but very useful trick: in the **first few hundred steps, ramp the LR up from very small to the target**, instead of starting at the large LR.

**为什么需要 warmup**（面试要点）：训练初期权重是随机的，梯度方向很不可靠；这时如果直接用大学习率，可能把模型推到一个糟糕的区域、甚至发散。warmup 让模型先"小步热身"、稳定下来，再用大学习率全速训练。**大 batch、Transformer、Adam 训练几乎都需要 warmup**。
**Why warmup is needed** (interview point): early on, weights are random and gradient directions unreliable; a large LR then can push the model into a bad region or diverge. Warmup lets the model "warm up with small steps" and stabilize before full-speed training. **Large batches, Transformers, and Adam training almost always need warmup.**

实务标配：**warmup + cosine**——先线性升到峰值，再余弦降到接近 0。下面手动构造这条曲线。
The standard combo: **warmup + cosine** — linear ramp to a peak, then cosine decay to near 0. We build it manually below.


In [ ]:
def warmup_cosine_lr(step, total, warmup, base_lr):
    if step < warmup:
        return base_lr * step / warmup                   # 线性升温阶段 / linear warmup
    # 之后余弦退火 / then cosine anneal to ~0
    progress = (step - warmup) / (total - warmup)
    return base_lr * 0.5 * (1 + np.cos(np.pi * progress))

total, warmup = 1000, 100
lrs = [warmup_cosine_lr(s, total, warmup, 0.1) for s in range(total)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lrs, lw=2)
ax.axvspan(0, warmup, alpha=0.12, color="orange", label="warmup(线性升温)")
ax.axvspan(warmup, total, alpha=0.08, color="green", label="cosine 退火")
ax.set_xlabel("训练步 step"); ax.set_ylabel("学习率 LR"); ax.legend()
ax.set_title("Warmup + Cosine: 先线性升温(稳住初期), 再余弦平滑降到~0(精细收敛)")
plt.tight_layout(); plt.show()
print("warmup: 初期权重随机/梯度不可靠 → 先小步热身防发散; 大batch/Transformer/Adam 几乎必用")
print("warmup+cosine 是现代训练(尤其 Transformer)的标准配方")


<a id="4"></a>
## 4. ReduceLROnPlateau ⭐ / ReduceLROnPlateau

前面的调度都是**预先设定好曲线**（按 epoch 走）。**ReduceLROnPlateau** 是**自适应**的：它**监控验证指标**（如验证损失），当指标**连续若干 epoch 不再改善**（进入"平台期"）时，才把学习率**乘以一个因子**降下来。
The schedules above follow a **preset curve** (by epoch). **ReduceLROnPlateau** is **adaptive**: it **monitors a validation metric** (e.g. val loss) and, when the metric **stops improving for several epochs** (a "plateau"), multiplies the LR by a factor.

优点：不用预先猜"该在第几轮降"——它根据训练实际情况决定。很适合不知道总训练轮数、或想"卡住了再降"的场景。和早停(7.3)是好搭档。
Advantage: no need to guess "at which epoch to drop" — it decides from actual training. Great when you don't know the total epochs or want to "drop only when stuck". A good partner to early stopping (7.3).


In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits(); X = digits.data/16.0
X_tr, X_te, y_tr, y_te = train_test_split(X, digits.target, test_size=0.3, stratify=digits.target, random_state=0)
Xtr_t = torch.tensor(X_tr, dtype=torch.float32); ytr_t = torch.tensor(y_tr)
Xte_t = torch.tensor(X_te, dtype=torch.float32); yte_t = torch.tensor(y_te)

torch.manual_seed(0)
net = nn.Sequential(nn.Linear(64,64), nn.ReLU(), nn.Linear(64,10))
opt = torch.optim.SGD(net.parameters(), lr=0.2)
# patience=5: 验证损失5轮不降就把 lr ×0.5 / drop LR when val loss plateaus
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=5)
ce = nn.CrossEntropyLoss()
lr_history, val_history = [], []
for epoch in range(80):
    opt.zero_grad(); ce(net(Xtr_t), ytr_t).backward(); opt.step()
    with torch.no_grad(): val = ce(net(Xte_t), yte_t).item()
    sched.step(val)                                       # 传入监控指标, 它决定要不要降 LR
    lr_history.append(opt.param_groups[0]["lr"]); val_history.append(val)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(val_history, "C0", label="验证损失"); ax1.set_xlabel("epoch"); ax1.set_ylabel("val loss", color="C0")
ax2 = ax1.twinx(); ax2.plot(lr_history, "C3", label="学习率"); ax2.set_ylabel("LR", color="C3")
ax1.set_title("ReduceLROnPlateau: 验证损失进入平台期 → 自动把 LR 减半")
plt.tight_layout(); plt.show()
print("验证损失卡住(平台期) → 自动把学习率乘 factor 降下来 → 常能再挤出一点提升")
print("优点: 自适应, 不用预设'第几轮降'; 适合未知总轮数; 和早停(7.3)是好搭档")


<a id="5"></a>
## 5. 实测：调度 vs 固定 + 小结 ⭐ / Scheduled vs Fixed & Summary

最后实测：**固定学习率 vs 带 cosine 调度**训同一个网络，对比最终效果。再提一个现代常用策略 **one-cycle**：学习率先升到一个较高峰值再降（同时动量反向变化），常能更快达到更好结果（fast.ai 推广）。
Finally: **fixed LR vs cosine schedule** on the same net, comparing final results. Plus a modern favorite, **one-cycle**: LR rises to a high peak then falls (with momentum varying inversely), often reaching better results faster (popularized by fast.ai).


In [ ]:
def train(scheduler_fn=None, lr=0.1, epochs=80):
    torch.manual_seed(0)
    net = nn.Sequential(nn.Linear(64,64), nn.ReLU(), nn.Linear(64,10))
    opt = torch.optim.SGD(net.parameters(), lr=lr, momentum=0.9)
    sched = scheduler_fn(opt) if scheduler_fn else None
    ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        opt.zero_grad(); ce(net(Xtr_t), ytr_t).backward(); opt.step()
        if sched: sched.step()
    return (net(Xte_t).argmax(1) == yte_t).float().mean().item()

acc_fixed = train(None)
acc_cosine = train(lambda o: torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=80))
# one-cycle: LR 先升到峰值再降 / one-cycle policy
acc_onecycle = train(lambda o: torch.optim.lr_scheduler.OneCycleLR(o, max_lr=0.3, total_steps=80))
print(f"固定学习率:        test 准确率 = {acc_fixed:.3f}")
print(f"Cosine 调度:       test 准确率 = {acc_cosine:.3f}")
print(f"One-Cycle 调度:    test 准确率 = {acc_onecycle:.3f}")
print("\n注: Digits 太小太简单, 三者差距很小(固定 LR 也不差); 调度的优势在大数据/深网络/长训练上才明显")
print("此处 one-cycle 略优; 实战标配: Adam/SGD + warmup + cosine, 或 SGD + one-cycle")


```
为什么调度: 初期离谷底远→大步(大LR), 后期近谷底→小步(小LR), 固定 LR 两头不讨好
常见: Step(阶梯骤降) / Exponential(指数) / Cosine(半余弦平滑, 现代常用)
Warmup: 初期权重随机/梯度不可靠 → 先线性升温防发散; 大batch/Transformer/Adam 必用
warmup+cosine: 现代训练标准配方(尤其 Transformer)
ReduceLROnPlateau: 自适应, 验证指标卡住才降 LR; 不用预设, 和早停搭配
One-Cycle: LR 先升到峰值再降; 常更快更好(fast.ai)
实战: Adam/SGD + warmup + cosine, 或 SGD + one-cycle
```

### 💡 面试速查 / Interview cheat-sheet
1. **学习率衰减**: 初期大(快)后期小(稳收敛); 固定 LR 两头不讨好。
   LR decay: large early (fast), small late (stable convergence); fixed LR is suboptimal both ends.
2. **warmup**: 初期梯度不可靠 → 先小步升温防发散(Transformer/大batch 必用)。
   Warmup: unreliable early gradients → ramp up to avoid divergence (must for Transformers/large batches).
3. **cosine annealing**: 半余弦平滑降到~0, 现代最常用之一。
   Cosine annealing: smooth half-cosine to ~0, a modern favorite.
4. **ReduceLROnPlateau**: 监控指标, 卡住才降(自适应)。
   ReduceLROnPlateau: monitor a metric, drop only on plateau (adaptive).
5. **One-Cycle**: 先升后降, 常更快更好(fast.ai)。
   One-Cycle: up then down, often faster and better.

### 下一节 / Next
**9.9 初始化**——权重初始值看似小事, 实则决定训练能否开始。糟糕的初始化会让信号在深层网络里爆炸或消失; Xavier/He 初始化解决它。
**9.9 Initialization** — weight init seems minor but decides whether training can even start. Bad init makes signals explode or vanish in deep nets; Xavier/He init fix it.
